# Imports

In [1]:
from idf_analysis.analysis.historical.bartlett_lewis import BartlettLewisModel, disaggregate_daily_to_subdaily
from idf_analysis.data.processing import read_csv, aggregate_to_csv, save_to_csv, verification, fill_missing_data

from idf_analysis.helpers.notebook import precip_summary
import pandas as pd

# Criação de precipitações sintéticas com Bartlett-Lewis

## Preparação de dados para a calibração

Obs.: Dados horários para calibração, dados diários para desagregação em 30 minutos

In [2]:
# Carregando dados horários para calibração
cemaden_hourly = read_csv(path='../results/cemaden_ac_santana_sao/cemaden_ac_santana_sao_hourly.csv')
inmet_hourly = read_csv(path='../results/inmet_sao_paulo_mirante/inmet_sao_paulo_mirante_hourly.csv')

summary_hourly = pd.concat([
    precip_summary(inmet_hourly, "INMET Horário"),
    precip_summary(cemaden_hourly, "CEMADEN Horário"),
])
print("📊 Dados horários para calibração: ")
display(summary_hourly.style.format(precision=2))

📊 Dados horários para calibração: 


,Total registros,Registros > 0,Primeira data,Última data,Precipitação média (mm),Precipitação máxima (mm)
INMET Horário,96432,7266,2014-01-01 00:00:00,2024-12-31 23:00:00,0.17,77.80
CEMADEN Horário,61305,6038,2014-01-01 00:00:00,2024-12-31 23:00:00,0.12,45.00


In [3]:
# Completar dados horários com fallback CEMADEN -> INMET
df_hourly_calib = fill_missing_data(
    df_main=cemaden_hourly,
	df_secondary=inmet_hourly,
    frequency='hourly'
)

# Verificação
verification(df=df_hourly_calib, frequency='hourly');

# Índice de timestamp
df_hourly_calib['timestamp'] = pd.to_datetime(df_hourly_calib[['Year', 'Month', 'Day', 'Hour']])
df_hourly_calib.set_index('timestamp', inplace=True)

print("✓ Dados horários completos para calibração:")
print(f"  {len(df_hourly_calib)} registros")
display(df_hourly_calib.head())

[OK] Série completa! Nenhum período faltando.

✓ Dados horários completos para calibração:
  96432 registros


,Year,Month,Day,Hour,Precipitation,Date
timestamp,,,,,,
2014-01-01 00:00:00,2014,1,1,0,0.0,2014-01-01 00:00:00
2014-01-01 01:00:00,2014,1,1,1,0.0,2014-01-01 01:00:00
2014-01-01 02:00:00,2014,1,1,2,0.0,2014-01-01 02:00:00
2014-01-01 03:00:00,2014,1,1,3,0.0,2014-01-01 03:00:00
2014-01-01 04:00:00,2014,1,1,4,0.0,2014-01-01 04:00:00


In [4]:
# Carregar dados diários para desagregação posterior
cemaden_daily = read_csv(path='../results/cemaden_ac_santana_sao/cemaden_ac_santana_sao_daily.csv')
inmet_daily = read_csv(path='../results/inmet_sao_paulo_mirante/inmet_sao_paulo_mirante_daily.csv')

df_daily = fill_missing_data(
    df_main=cemaden_daily,
	df_secondary=inmet_daily,
    frequency='daily'
)

df_daily['timestamp'] = pd.to_datetime(df_daily[['Year', 'Month', 'Day']])
df_daily.set_index('timestamp', inplace=True)

print(f"✓ Dados diários completos para desagregação: {len(df_daily)} registros")
display(df_daily.head())

✓ Dados diários completos para desagregação: 4018 registros


,Year,Month,Day,Precipitation,Date
timestamp,,,,,
2014-01-01,2014,1,1,1.18,2014-01-01
2014-01-02,2014,1,2,2.18,2014-01-02
2014-01-03,2014,1,3,0.00,2014-01-03
2014-01-04,2014,1,4,1.58,2014-01-04
2014-01-05,2014,1,5,17.85,2014-01-05


In [5]:
# Inicializar modelo
bl_model = BartlettLewisModel()
print("✓ Modelo inicializado")

✓ Modelo inicializado


## Inicialização do modelo e identificação de eventos

In [6]:
# Identificar eventos com gap de 4 horas (240 minutos)
print("\n🔍 Identificando eventos de precipitação...")
events = bl_model.identify_events(
	df_hourly_calib['Precipitation'],
	inter_event_gap_minutes=240  # 4 horas
)

print(f"   ✓ {len(events)} eventos identificados")


🔍 Identificando eventos de precipitação...
   ✓ 2354 eventos identificados


## Calibração do modelo

In [7]:
print("\n⚙️  Calibrando modelo com dados horários...")
params = bl_model.calibrate(
    events=events,
    interval_minutes=60,  # Dados horários têm intervalo de 60 min
    intra_event_gap_minutes=60,  # Período seco entre pulsos: 1 hora
)

print("\n📊 Parâmetros Calibrados:")
for key, value in params.items():
	print(f"   {key:8s}: {value:.4f}")

# Exportar parâmetros
output_file = '../results/cemaden_ac_santana_sao/bartlett_lewis_params.yaml'
bl_model.export_params(output_file)
print(f"\n✓ Parâmetros salvos em: {output_file}")


⚙️  Calibrando modelo com dados horários...

📊 Parâmetros Calibrados:
   lambda  : 0.5862
   beta    : 1.0000
   gamma   : 0.0015
   eta     : 0.0015
   mu      : 0.0107
✅ Parâmetros exportados para: ../results/cemaden_ac_santana_sao/bartlett_lewis_params.yaml

✓ Parâmetros salvos em: ../results/cemaden_ac_santana_sao/bartlett_lewis_params.yaml


## Geração da chuva sintética com o modelo

In [8]:
print("\n🌧️  Desagregando chuva diária para intervalos de 30 minutos...")
synthetic_rain = bl_model.disaggregate(
	coarse_series=df_daily['Precipitation'],
	fine_interval_minutes=30,  # 30 minutos
	seed=None
)
synthetic_rain.to_csv('../results/bl_chuva_sintetica.csv')
print("✓ Chuva sintética desagregada e salva.")


🌧️  Desagregando chuva diária para intervalos de 30 minutos...
✓ Chuva sintética desagregada e salva.


# Agregação da chuva sintética

### Formatação dos dados para utilização da função de agregação

In [9]:
# Separa a coluna timestamp em Year, Month, Day, Hour e Minute
st_rain_formatted = synthetic_rain.reset_index()
st_rain_formatted.columns = ['timestamp', 'rainfall_mm']
st_rain_formatted['Year'] = st_rain_formatted['timestamp'].dt.year
st_rain_formatted['Month'] = st_rain_formatted['timestamp'].dt.month
st_rain_formatted['Day'] = st_rain_formatted['timestamp'].dt.day
st_rain_formatted['Hour'] = st_rain_formatted['timestamp'].dt.hour
st_rain_formatted['Minute'] = st_rain_formatted['timestamp'].dt.minute

# Renomeia a coluna rainfall_mm para Precipitation
st_rain_formatted = st_rain_formatted[['Year', 'Month', 'Day', 'Hour', 'Minute', 'rainfall_mm']]
st_rain_formatted.columns = ['Year', 'Month', 'Day', 'Hour', 'Minute', 'Precipitation']


In [10]:
st_rain_formatted.head()

,Year,Month,Day,Hour,Minute,Precipitation
0,2014,1,1,0,0,0.024583
1,2014,1,1,0,30,0.024583
2,2014,1,1,1,0,0.024583
3,2014,1,1,1,30,0.024583
4,2014,1,1,2,0,0.024583


In [11]:
bl_rain_name = 'cemaden_ac_santana_sao_synthetic'
output_path = '../results/cemaden_ac_santana_sao/'

# Agregar para resolução diária
aggregate_to_csv(
	st_rain_formatted,
	name=bl_rain_name,
	directory=output_path,
	include_minutes=False,  # Agregação apenas para hora completa
)
print(f"\n✓ Chuva sintética horária salva em: {output_path}{bl_rain_name}_hourly.csv")


✓ Chuva sintética horária salva em: ../results/cemaden_ac_santana_sao/cemaden_ac_santana_sao_synthetic_hourly.csv


In [12]:
df_synthetic_hourly = read_csv(f"{output_path}{bl_rain_name}_hourly.csv")
df_synthetic_hourly.head()

,Year,Month,Day,Hour,Precipitation
0,2014,1,1,0,0.049167
1,2014,1,1,1,0.049167
2,2014,1,1,2,0.049167
3,2014,1,1,3,0.049167
4,2014,1,1,4,0.049167


# Comparação da chuva sintética com a chuva original

In [13]:
# Validação: Comparação de volumes totais entre dados originais e desagregados
print("\nVALIDAÇÃO: Comparação de Volumes Totais")
print("="*60)

summary_hourly = pd.concat([
    precip_summary(df_hourly_calib, "Dados Horários Calibração"),
    precip_summary(df_synthetic_hourly, "Dados Horários Bartlett-Lewis"),
])
display(summary_hourly.style.format(precision=2))

# Volume total horário original
volume_original = df_hourly_calib['Precipitation'].sum()
print(f"\nVolume total CEMADEN (horário): {volume_original:.2f} mm")

# Volume total desagregado (horário)
volume_desagregado = df_synthetic_hourly['Precipitation'].sum()
print(f"Volume total Sintético (horário): {volume_desagregado:.2f} mm")

# Diferença percentual
diferenca = abs(volume_original - volume_desagregado) / volume_original * 100
print(f"\nDiferença: {diferenca:.4f}%")


VALIDAÇÃO: Comparação de Volumes Totais


,Total registros,Registros > 0,Primeira data,Última data,Precipitação média (mm),Precipitação máxima (mm)
Dados Horários Calibração,96432,26544,2014-01-01 00:00:00,2024-12-31 23:00:00,0.14,45.00
Dados Horários Bartlett-Lewis,96432,47754,2014-01-01 00:00:00,2024-12-31 23:00:00,0.12,56.46



Volume total CEMADEN (horário): 13880.80 mm
Volume total Sintético (horário): 11784.33 mm

Diferença: 15.1034%


In [14]:
from idf_analysis.analysis.historical.subdaily import get_max_subdaily_table

# Comparacao dos maximos subdiarios (horarios) entre original e BL
output_dir_comp = "../results/cemaden_bl_comparison"
orig_name = "cemaden_ac_santana_sao"
syn_name = "cemaden_ac_santana_sao_bartlett_lewis_synthetic"

max_subdaily_original = get_max_subdaily_table(
    df=df_hourly_calib,
    name_file=orig_name,
    output_dir=output_dir_comp,
)

max_subdaily_synthetic = get_max_subdaily_table(
    df=df_synthetic_hourly,
    name_file=syn_name,
    output_dir=output_dir_comp,
)

print("\n📊 Original Calibração - Máximos subdiários:")
display(max_subdaily_original)

print("\n📊 Bartlett-Lewis - Máximos subdiários:")
display(max_subdaily_synthetic)	


[OK] Resultados salvos em: ../results/cemaden_bl_comparison/max_subdaily_cemaden_ac_santana_sao.csv

[OK] Resultados salvos em: ../results/cemaden_bl_comparison/max_subdaily_cemaden_ac_santana_sao_bartlett_lewis_synthetic.csv

📊 Original Calibração - Máximos subdiários:


,Year,Max_1h,Max_3h,Max_6h,Max_8h,Max_10h,Max_12h,Max_24h
0,2014,45.00,55.46,63.04,64.22,67.36,68.36,119.54
1,2015,43.99,89.54,94.08,94.28,94.48,94.48,104.33
2,2016,25.82,40.83,52.29,59.26,61.19,61.34,65.00
3,2017,26.84,55.56,67.17,67.97,68.35,70.52,87.44
4,2018,12.20,17.70,26.50,34.36,42.61,49.68,74.89
5,2019,25.98,31.88,34.25,36.14,36.14,36.34,46.46
6,2020,12.40,17.54,32.20,32.85,33.09,40.44,50.97
7,2021,11.14,21.41,29.53,37.09,42.20,44.06,57.36
8,2022,37.38,44.45,44.65,44.65,44.65,44.65,62.58
9,2023,30.74,41.03,43.01,43.81,43.81,44.20,56.44



📊 Bartlett-Lewis - Máximos subdiários:


,Year,Max_1h,Max_3h,Max_6h,Max_8h,Max_10h,Max_12h,Max_24h
0,2014,16.92,35.76,65.56,65.56,67.89,68.92,75.91
1,2015,14.41,31.30,62.60,83.46,104.33,104.36,104.56
2,2016,25.76,50.31,50.31,50.31,50.31,50.31,84.02
3,2017,56.46,84.69,84.69,84.69,84.69,84.69,86.86
4,2018,6.01,14.93,23.71,29.56,35.40,41.25,76.34
5,2019,4.95,12.37,20.68,24.30,24.66,25.00,35.94
6,2020,9.39,21.59,21.59,21.59,21.59,21.59,34.76
7,2021,5.05,13.56,15.82,15.82,15.83,15.83,30.92
8,2022,25.95,27.54,29.92,31.51,33.10,34.69,58.24
9,2023,11.84,13.94,27.88,30.46,31.00,31.53,55.01
